# Using Tools With Claude

This notebook demonstrates how tools let an AI assistant gather external information before answering. We will build a small mock weather tool and walk through the complete tool-calling message flow.

## What You Will Learn

- How to call Claude with plain text and structured messages
- How to define a tool with typed parameters
- How the model requests a tool call
- How to return tool results to the model
- How the final assistant response is produced

## Setup

Load environment variables, clear cached course helper modules, then import the local wrapper classes. The Anthropic SDK will use its default credential lookup, including `ANTHROPIC_API_KEY` from your environment or `.env` file.

In [1]:
import json
import os
import sys

from dotenv import load_dotenv

load_dotenv(".env")
# Refresh local course helpers after editing files in lib/.
for module_name in [name for name in list(sys.modules) if name == "lib" or name.startswith("lib.")]:
    del sys.modules[module_name]

from lib.llm import LLM
from lib.messages import SystemMessage, ToolMessage, UserMessage
from lib.tooling import tool

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
chat_model = LLM(provider="claude_code", model="claude-sonnet-4-6", timeout=30)
print(f"Ready: {chat_model.model}")

## Basic LLM Interaction

Before adding tools, start with two standard interaction patterns: a single text prompt and a structured conversation made from message objects.

In [ ]:
response = chat_model.invoke("What is an AI agent?")
print(response.content)

In [ ]:
messages = [
    SystemMessage(content="You're an Anthropic Claude specialist."),
    UserMessage(content="What is function calling?"),
]

response = chat_model.invoke(messages)
print(response.content)

## Build A Tool

A tool is a normal Python function wrapped with `@tool`. The wrapper converts it into the function schema the model receives so it can decide when to call it.

In [ ]:
@tool
def get_weather(city: str):
    """Get the current temperature for a city.

    Args:
        city: Name of the city to check weather for.

    Returns:
        A dictionary containing temperature information for the requested city.
    """
    mock_weather = {
        "São Paulo": "28°C",
        "Oslo": "-3°C",
        "New York": "15°C",
        "Tokyo": "22°C",
    }
    return {"temperature": mock_weather.get(city, "Unknown")}

In [ ]:
chat_model_with_tools = LLM(provider="claude_code", model="claude-sonnet-4-6", tools=[get_weather], timeout=30)

## Tool Usage Flow

The model does not execute your local Python function by itself. It returns a tool-call request, your code runs the tool, then your code sends a `ToolMessage` back to the model.

The flow is:

1. The user asks a weather question.
2. The assistant responds with a tool call.
3. Your code executes the requested tool.
4. Your code appends a `ToolMessage` with the result.
5. The assistant uses the tool result to produce the final answer.

In [ ]:
messages = [
    SystemMessage(
        content=(
            "You are a helpful assistant that can get current temperatures for cities. "
            "Use the weather tool when someone asks about weather or temperature "
            "in a specific location. If the tool does not know, say so."
        )
    ),
    UserMessage(content="How cold is it in Oslo?"),
]

In [ ]:
ai_message = chat_model_with_tools.invoke(messages)
messages.append(ai_message)

ai_message

In [ ]:
tool_call = ai_message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

tool_result = get_weather(**args)
tool_message = ToolMessage(
    content=tool_result["temperature"],
    tool_call_id=tool_call.id,
    name=tool_call.function.name,
)
messages.append(tool_message)

tool_message

In [ ]:
final_message = chat_model_with_tools.invoke(messages)
messages.append(final_message)

final_message.content

## Inspect The Conversation

The full message list shows each step: system instruction, user request, assistant tool call, tool response, and final assistant answer.

In [ ]:
messages